## Overview

This notebook loads the pickled results produced by `scaling_effdim_dimIn.py`, which scans the input-basis dimension D (dimension of the correlation spectrum S in Gamma = U @ diag(S) @ V^T) while holding the correlation-spectrum purity tr(S^4) fixed at `purity0`, and plots the normalized effective dimension against the ratio D/M (input dimension over number of parameters), for two combinations of number of parameters M and local basis dimension Dloc (d_tilde).

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import pickle
import numpy

import pennylane.numpy as np




import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the fixed scan parameters (target correlation-spectrum purity tr(S^4) = `purity0`, number of parameter samples, number of random V realizations per input dimension) and lists the (Dloc, M) combinations whose pickled input-dimension scans (over `dim_in_vec`, produced by `scaling_effdim_dimIn.py`) will be loaded and overlaid.

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_dimInSpace/'

# Fixed purity of S
purity0 = 0.333
purity0_name = '0p333'

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 150
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 50
no_V_samples_name = str(no_matrix_realiz)

########################### Exps. to plot ###########################

# Local dimension of parameter functions space
local_dim_param_vec = [2, 3]
name_dim_par_loc_vec = [str(local_dim_param) for local_dim_param in local_dim_param_vec]

# Dimension of input function space
no_params_vec = [10, 7]
name_no_params_vec = [str(no_params) for no_params in no_params_vec]

no_exps = len(local_dim_param_vec)

For each (Dloc, M) combination, loads and concatenates the pickled input dimension, correlation-spectrum purity tr(S^4), and normalized effective dimension arrays across all matching result files into `dict_exps`.

In [ ]:
dict_exps = dict()

for i in range(no_exps):
    local_dim_param = local_dim_param_vec[i]
    name_dim_par_loc = name_dim_par_loc_vec[i]
    no_params = no_params_vec[i]
    name_no_params = name_no_params_vec[i]

    namefileend = ('_DimLocPar' + name_dim_par_loc + '_Nparams' + name_no_params + '_NsamplesPar' + no_par_samples_name + 
                   '_NsamplesV' + no_V_samples_name + '_S4trace' + purity0_name + '.pkl')
    
    namefileini = 'norm_eff_dim'
    listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini) and
                                                              f.endswith(namefileend))]
    print(len(listallfiles))
    
    dim_in_all = []
    purity_S_all = []
    norm_eff_dim_all = []
    
    for filename in listallfiles:
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
    
        dimin_vec = dict_norm_ed['dim_in_all']
        purity_S_vec = dict_norm_ed['purity_S_all']
        norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
    
        dim_in_all.append(dimin_vec)
        purity_S_all.append(purity_S_vec)
        norm_eff_dim_all.append(norm_eff_dim_vec)
    
    dim_in_all = np.concatenate(dim_in_all)
    purity_S_all = np.concatenate(purity_S_all)
    norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

    dict_exp = dict()
    dict_exp['local_dim_param'] = local_dim_param
    dict_exp['no_params'] = no_params
    dict_exp['purity_S_all'] = purity_S_all
    dict_exp['dim_in_all'] = dim_in_all
    dict_exp['norm_eff_dim_all'] = norm_eff_dim_all
    dict_exps[i] = dict_exp

Plots the normalized effective dimension d_eff against D/M (log-scaled x-axis) for the two configurations, labeling each curve with M, Dloc (d_tilde), and the (fixed) correlation-spectrum purity tr(S^4).

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange']
for i in range(no_exps):
    dict_exp = dict_exps[i]
    no_params = dict_exp['no_params']
    local_dim_param = dict_exp['local_dim_param']
    dim_in_all = dict_exp['dim_in_all']
    norm_eff_dim_all = dict_exp['norm_eff_dim_all']
    purity_S_all = numpy.asarray(dict_exp['purity_S_all'])
    purity_rounded = round(numpy.mean(purity_S_all), 2)
    print('S purity: ', purity_rounded, ' with std. ', np.std(purity_S_all))
    lbl = r'$M=' + str(no_params) + ',\,' + r'\tilde{d}=' + str(local_dim_param) + ',\,\mathrm{tr}\,S^4=' + str(purity_rounded) + '$'
    x = np.asarray(dim_in_all) / no_params
    y = np.asarray(norm_eff_dim_all)
    ax.plot(x, y, '.', color=clrs[i], markersize=12, label=lbl)

#ax.legend(fontsize=22, loc='lower left')
ax.legend(fontsize=22)
ax.set_xlabel(r'$D/M$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])

fig.savefig('EffDimScaling_vs_D.png', bbox_inches='tight')
fig.savefig('EffDimScaling_vs_D.pdf', bbox_inches='tight')